### Create Synthetic Data for Fine-Tuning

The pipe consists of two components:

1. A Large Vision-Langauage model to generate QA pairs
2. A separate VLM to evaluate and filter those pairs


The Setup:

* **Teacher model**: microsoft/Phi-4-multimodal-instruct - We use a powerful 5.6B parameter instruction-tuned vision-language model (VLM) to generate high-quality question-answer (QA) data.
* **Jodge model**: Qwen/Qwen2.5-VL-7B-Instruct - A 7B parameter vision-language model used to assess both the relevance and correctness of each QA pair.
* **Optional: Huamn in the loop Verification**


#### Step 1: Load the QA Generator Model

In [9]:
from transformers import AutoTokenizer, AutoModelForVision2Seq, AutoProcessor
import torch


smol_gen_id = "HuggingFaceTB/SmolVLM-Instruct"
smol_gen_tokenizer = AutoTokenizer.from_pretrained(smol_gen_id)
smol_processor = AutoProcessor.from_pretrained(smol_gen_id, trust_remote_code=True)
smol_gen_model = AutoModelForVision2Seq.from_pretrained(
    smol_gen_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",  
)

tokenizer_config.json:   0%|          | 0.00/4.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

...

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [21]:
from PIL import Image
from transformers.image_utils import load_image
def generate_qa(image: Image.Image) -> str:

    prompt = (
        "<image>\n"  # Add image token here
        "You are an AI that sees the image and explains it to a human.\n"
        "Describe the image in detail, including objects, actions, and any text present.\n"
    )

    inputs = smol_processor(
        text=prompt,
        images=image,
        return_tensors="pt",
    ).to(smol_gen_model.device)

    outputs = smol_gen_model.generate(
        **inputs, max_new_tokens=500
    )
    output_text = smol_processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return output_text


image = load_image("/Users/aakinlalu/pet_project/GenerativeAI/images_dir/2491f5167f83dbea39f8f5f3b5cdc2e1.jpg") 
print(generate_qa(image))


<image>You are an AI that sees the image and explains it to a human.
Describe the image in detail, including objects, actions, and any text present.

The image shows a modern entertainment center in a home. The center is made of dark wood and features a large TV mounted on the wall. The TV is surrounded by a black frame and has a soundbar below it. The entertainment center is also decorated with shelves and cabinets, which are filled with books, DVDs, and other items. The shelves are made of the same dark wood as the rest of the entertainment center. The cabinets are black and have a modern design. The floor is covered in light gray carpet. The ceiling is white and has recessed lighting. The walls are painted white. The entertainment center is a large piece of furniture that is used to display and store various items. The entertainment center is a popular piece of furniture in modern homes. The entertainment center is a large piece of furniture that is used to display and store various

#### Step 2: Load the VLM Judge Model
Load a model to act as a “judge” that filters out low-quality QA pairs

In [4]:
from transformers import AutoModelForVision2Seq

qwen_id = "Qwen/Qwen2.5-VL-7B-Instruct"
qwen_processor = AutoProcessor.from_pretrained(qwen_id)
qwen_model = AutoModelForVision2Seq.from_pretrained(
    qwen_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

def judge_qa(image: Image.Image, question:str, answer:str) -> str:
    judge_prompt = (
        f"<img>\nYou are an expert evaluator. Assess if the answer is correct and relevant to the question.\n"
        f"Question: {question}\nAnswer: {answer}\n\n"
        f"Reply with only 'yes' or 'no'."
    )

    inputs = qwen_processor(
        text=judge_prompt,
        images=image,
        return_tensors="pt",
    ).to(qwen_model.device)
    outputs = qwen_model.generate(
        **inputs, max_new_tokens=10
    )
    
    output_text = qwen_processor.batch_decode(outputs, skip_special_tokens=True)[0].lower()
    return "yes" in output_text


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [22]:
import os
image_folder = "/Users/aakinlalu/pet_project/GenerativeAI/images_dir"  # Replace with your image folder path
accepted_data = []


def parse_qa(text):
    if "Question:" in text and "Answer:" in text:
        question = text.split("Question:")[1].split("Answer:")[0].strip()
        answer = text.split("Answer:")[1].strip()
        return question, answer
    return None, None

for image_path in os.listdir(image_folder):
    if image_path.endswith(('.jpg', '.jpeg', '.png')):
        image = load_image(os.path.join(image_folder, image_path))

        print(f"Processing image: {image}")
        
        # Generate question and answer
        generated_text = generate_qa(image)
        question, answer = parse_qa(generated_text)
        
        if question and answer:
            print(f"Generated Question: {question}")
            print(f"Generated Answer: {answer}")
            
            # Judge the QA
            is_correct = judge_qa(image, question, answer)
            print(f"Is the answer correct? {'Yes' if is_correct else 'No'}")
            
            if is_correct:
                accepted_data.append({
                    "image_path": image_path,
                    "question": question,
                    "answer": answer
                })
        else:
            print("Failed to parse generated text.")

Processing image: <PIL.Image.Image image mode=RGB size=736x1104 at 0x377D88510>
Failed to parse generated text.
Processing image: <PIL.Image.Image image mode=RGB size=528x792 at 0x178EF5050>
Failed to parse generated text.
Processing image: <PIL.Image.Image image mode=RGB size=736x876 at 0x17943E6D0>
Failed to parse generated text.
